In [6]:
import pandas as pd
import time
import requests
import argparse
from nba_api.stats.endpoints import ScoreboardV3, BoxScoreTraditionalV3
from datetime import datetime, timedelta

# ==========================================
# CONFIGURATION & HEADERS & REACHABILITY
# ==========================================
custom_headers = {
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com',
}

TEAM_HASHTAGS = {
    'ATL': '#TrueToAtlanta',
    'BKN': '#NetsWorld',
    'BOS': '#DifferentHere',
    'CHA': '#LetsFly',
    'CHI': '#SeeRed',
    'CLE': '#LetEmKnow',
    'DAL': '#MFFL',
    'DEN': '#MileHighBasketball',
    'DET': '#DetroitBasketball',
    'GSW': '#DubNation',
    'HOU': '#Rockets',
    'IND': '#BoomBaby',
    'LAC': '#ClipperNation',
    'LAL': '#LakeShow',
    'MEM': '#GrindCity',
    'MIA': '#HEATCulture',
    'MIL': '#FearTheDeer',
    'MIN': '#RaisedByWolves',
    'NOP': '#Pelicans',
    'NYK': '#NewYorkForever',
    'OKC': '#ThunderUp',
    'ORL': '#MagicTogether',
    'PHI': '#BrotherlyLove',
    'PHX': '#ValleyProud',
    'POR': '#RipCity',
    'SAC': '#LightTheBeam',
    'SAS': '#PorVida',
    'TOR': '#WeTheNorth',
    'UTA': '#TakeNote',
    'WAS': '#ForTheDistrict'
}

# ==========================================
# TEST MODE: MOCK DATA GENERATORS
# ==========================================
def get_mock_games():
    """Returns static, offline game data to bypass NBA API blocks. Proving the code works"""
    print("[TEST MODE] Loading mock scoreboard data...")
    mock_games = pd.DataFrame([{
        'GAME_ID': '0022300675', 
        'GAME_STATUS_TEXT': 'Final',
        'HOME_TEAM_ID': 1610612739, # Cleveland Cavaliers
        'VISITOR_TEAM_ID': 1610612765 # Detroit Pistons
    }])
    mock_lines = pd.DataFrame([
        {'GAME_ID': '0022300675', 'TEAM_ID': 1610612739, 'TEAM_ABBREVIATION': 'CLE', 'PTS': 128},
        {'GAME_ID': '0022300675', 'TEAM_ID': 1610612765, 'TEAM_ABBREVIATION': 'DET', 'PTS': 121}
    ])
    return mock_games, mock_lines

def get_mock_box_score():
    """Returns offline box score data."""
    print("[TEST MODE] Loading mock box scores...")
    return pd.DataFrame([
        {'nameI': 'D. Mitchell', 'plusMinusPoints': 22.0, 'points': 45},
        {'nameI': 'J. Allen', 'plusMinusPoints': 19.0, 'points': 14},
        {'nameI': 'C. Cunningham', 'plusMinusPoints': -12.0, 'points': 19},
        {'nameI': 'J. Ivey', 'plusMinusPoints': -15.0, 'points': 14}
    ])

# ==========================================
# LIVE MODE: API FUNCTIONS (WITH BACKOFF)
# ==========================================
def fetch_completed_games(max_retries=3):
    print("Fetching live Scoreboard data...")
    for attempt in range(max_retries):
        try:
            target_date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
            scoreboard = ScoreboardV3(game_date=target_date, headers=custom_headers, timeout=30)
            games = scoreboard.game_header.get_data_frame()
            line_score = scoreboard.line_score.get_data_frame()
            
            if games.empty:
                return pd.DataFrame(), pd.DataFrame()

            completed_games = games[games['GAME_STATUS_TEXT'].str.contains('Final', na=False, case=False)].copy()
            return completed_games, line_score

        except (requests.exceptions.RequestException, TimeoutError) as e:
            print(f"Scoreboard request failed on attempt {attempt + 1}: {e}")
            time.sleep(2 ** attempt)
            
    print("Max retries reached. NBA API is currently blocking requests.")
    return pd.DataFrame(), pd.DataFrame()

def get_box_scores(game_id, max_retries=3):
    for attempt in range(max_retries):
        try:
            box_score = BoxScoreTraditionalV3(game_id=game_id, headers=custom_headers, timeout=30)
            return box_score.get_data_frames()[0]
        except (requests.exceptions.RequestException, TimeoutError):
            time.sleep(2 ** attempt)
            
    return pd.DataFrame()

# ==========================================
# MAIN PIPELINE (ETL)
# ==========================================
def extract_stats(is_test_mode=False):
    # Route to mock data if test mode is enabled, otherwise hit live API
    if is_test_mode:
        completed_games, line_score = get_mock_games()
    else:
        completed_games, line_score = fetch_completed_games()

    if completed_games.empty or line_score.empty:
        print("Pipeline aborted: No data available.")
        return

    for _, game in completed_games.iterrows():
        game_id = str(game.get('GAME_ID', '')).strip()
        
        home_row = line_score[line_score['TEAM_ID'] == game.get('HOME_TEAM_ID')].iloc[0]
        visitor_row = line_score[line_score['TEAM_ID'] == game.get('VISITOR_TEAM_ID')].iloc[0]

        home_team = home_row.get('TEAM_ABBREVIATION', '')
        home_score = home_row.get('PTS', 0)
        visitor_team = visitor_row.get('TEAM_ABBREVIATION', '')
        away_score = visitor_row.get('PTS', 0)

        box_scores = get_mock_box_score() if is_test_mode else get_box_scores(game_id)
        
        if box_scores.empty or 'plusMinusPoints' not in box_scores.columns:
            continue

        valid_box_scores = box_scores.dropna(subset=['plusMinusPoints'])
        best_player = valid_box_scores.loc[valid_box_scores['plusMinusPoints'].idxmax()]
        worst_player = valid_box_scores.loc[valid_box_scores['plusMinusPoints'].idxmin()]
        home_tag = TEAM_HASHTAGS.get(home_team, f"#{home_team}")
        away_tag = TEAM_HASHTAGS.get(visitor_team, f"#{visitor_team}")
        matchup_tag = f"#{visitor_team}vs{home_team}"

        tweet = (
            f"🏀 Final Score: {home_team} {home_score} - {away_score} {visitor_team}\n\n"
            f"Best +/-: {best_player['nameI']} (+{int(best_player['plusMinusPoints'])})\n"
            f"Worst +/-: {worst_player['nameI']} ({int(worst_player['plusMinusPoints'])})\n"
            f"{matchup_tag} {home_tag} {away_tag} #NBA"
        )
        print("-" * 40)
        print(tweet)
        print("-" * 40)

if __name__ == "__main__":
    # Running in test mode to guarantee portfolio demonstration output
    extract_stats(is_test_mode=True)

[TEST MODE] Loading mock scoreboard data...
[TEST MODE] Loading mock box scores...
----------------------------------------
🏀 Final Score: CLE 128 - 121 DET

Best +/-: D. Mitchell (+22)
Worst +/-: J. Ivey (-15)
#DETvsCLE #LetEmKnow #DetroitBasketball #NBA
----------------------------------------
